In [ ]:
!pip install pretty_midi
!pip install pyworld
!pip install noisereduce

#MIDI파일에서 vocal부분만 추출

In [ ]:
import pretty_midi

def extract_vocal_track(midi_path, save_path="vocal_only.mid"):
    midi = pretty_midi.PrettyMIDI(midi_path)

    # 1) 트랙 이름 기반 탐색
    vocal_keywords = ['vocal','voice','melody','lead','main','lyric','vox','sing']
    vocal_instr = None

    for inst in midi.instruments:
        name = inst.name.lower()
        if any(kw in name for kw in vocal_keywords):
            vocal_instr = inst
            break

    # 2) 없다면 자동 추정 (싱글톤 멜로디 기반)
    if vocal_instr is None:
        print("[!] No vocal-named track found → Guessing melody track...")

        best_inst = None
        best_score = -1

        for inst in midi.instruments:
            # Drum은 제외
            if inst.is_drum:
                continue

            notes = inst.notes
            if len(notes) == 0:
                continue

            # 멜로디성 점수 계산
            # 단음 위주인지 평가 (chord 비율이 낮을수록 좋음)
            pitches = [n.pitch for n in notes]
            unique_ratio = len(set(pitches)) / len(pitches)

            # 사람 음역대 체크
            in_vocal_range = sum(1 for n in notes if 48 < n.pitch < 84)

            # 점수 구성
            score = unique_ratio * 0.5 + in_vocal_range * 0.5

            if score > best_score:
                best_score = score
                best_inst = inst

        vocal_instr = best_inst

    # 3) 새로운 MIDI 구성
    new_midi = pretty_midi.PrettyMIDI()
    new_instr = pretty_midi.Instrument(program=0, name="VOCAL_EXTRACTED")
    new_instr.notes = vocal_instr.notes
    new_midi.instruments.append(new_instr)
    new_midi.write(save_path)

    print(f"[완료] Vocal-only MIDI saved → {save_path}")


# 사용법
extract_vocal_track("/content/attasanchi_op.mid", "vocal_only.mid")


#MIDI파일에 메트노롬 추가
이 메트노롬이 추가된 음성을 듣고 녹음은 따로 해야한다.

In [ ]:
import pretty_midi
import numpy as np
from scipy.io.wavfile import write

class GuideAudioGenerator:
    def __init__(self, midi_path, fs=44100):
        self.fs = fs
        self.midi_path = midi_path

        print(f"[초기화] MIDI 분석 중: {self.midi_path}")
        self.prepare_guide_audio()

    def prepare_guide_audio(self):
        midi_data = pretty_midi.PrettyMIDI(self.midi_path)

        # BPM
        try:
            tempo = midi_data.get_tempo_changes()[1][0]
        except:
            tempo = 120.0

        beat_sec = 60.0 / tempo
        beat_samples = int(beat_sec * self.fs)
        print(f"-> BPM: {tempo:.2f}")

        # ---------------------------------------------------
        # 1. MIDI 합성
        # ---------------------------------------------------
        y_midi = midi_data.synthesize(fs=self.fs)
        total_samples = len(y_midi)

        # ---------------------------------------------------
        # 2. 메트로놈 생성 (클릭 톤)
        # ---------------------------------------------------
        click = self.generate_click_tone()
        click_len = len(click)

        y_click = np.zeros(total_samples)
        for i in range(0, total_samples - click_len, beat_samples):
            y_click[i:i+click_len] += click

        # ---------------------------------------------------
        # 3. 메인 가이드 오디오
        # ---------------------------------------------------
        main_audio = (y_midi * 0.5) + (y_click * 0.8)

        # ---------------------------------------------------
        # 4. 예비박(Count-in) 4박자 추가
        # ---------------------------------------------------
        pre_roll = np.zeros(4 * beat_samples)
        for i in range(4):
            idx = i * beat_samples
            pre_roll[idx:idx+click_len] += click

        # 전체 가이드: 예비박 + 본문
        self.full_guide = np.concatenate((pre_roll, main_audio))

        # ---------------------------------------------------
        # 5. 볼륨 정규화
        # ---------------------------------------------------
        max_val = np.max(np.abs(self.full_guide))
        if max_val > 0:
            self.full_guide = self.full_guide / max_val * 0.9

        print("🎧 가이드 오디오 생성 완료!")

    def generate_click_tone(self):
        """메트로놈 클릭음"""
        t = np.arange(int(0.05 * self.fs)) / self.fs
        tone = np.sin(2 * np.pi * 1000 * t) * np.exp(-10 * t)
        return tone

    def save(self, filename):
        """WAV 파일 저장"""
        audio_int16 = (self.full_guide * 32767).astype(np.int16)
        write(filename, self.fs, audio_int16)
        print(f"✅ 가이드 오디오 저장 완료: {filename}")

    def get_audio(self):
        """numpy 배열 반환"""
        return self.full_guide

#pitch shift

In [ ]:
import numpy as np
import librosa
import soundfile as sf
import pretty_midi
import pyworld as pw
import noisereduce as nr
from scipy.signal import butter, filtfilt

# --- 필터 함수 (기존 동일) ---
def highpass_filter(data, cutoff, fs, order=5):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, data)

def lowpass_filter(data, cutoff, fs, order=5):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

def process_sts_safe(audio_path, midi_path, output_path, fs=44100):
    print("=== [1] 데이터 로드 및 전처리 ===")

    y, _ = librosa.load(audio_path, sr=fs, dtype=np.float64)

    # 노이즈 리덕션
    noise_len = int(0.5 * fs)
    noise_part = y[-noise_len:] if len(y) > noise_len else y
    y_denoised = nr.reduce_noise(y=y, sr=fs, y_noise=noise_part, prop_decrease=0.8, stationary=True)
    y = np.ascontiguousarray(y_denoised, dtype=np.float64)

    # MIDI 로드 및 길이 맞춤
    midi_data = pretty_midi.PrettyMIDI(midi_path)
    midi_duration = midi_data.get_end_time()

    # 목소리 정렬 (Trim)
    y_trimmed, _ = librosa.effects.trim(y, top_db=30)

    # MIDI 시작점 찾기 (앞부분 공백 처리)
    first_note_time = 0.0
    for instrument in midi_data.instruments:
        if not instrument.is_drum:
            for note in instrument.notes:
                if note.start < first_note_time or first_note_time == 0.0:
                    first_note_time = note.start

    silence_pad = np.zeros(int(first_note_time * fs), dtype=np.float64)
    y_aligned = np.concatenate((silence_pad, y_trimmed))

    target_samples = int(midi_duration * fs)
    if len(y_aligned) > target_samples:
        y_processed = y_aligned[:target_samples]
    else:
        pad_width = target_samples - len(y_aligned)
        y_processed = np.pad(y_aligned, (0, pad_width), mode='constant')

    print("=== [2] World 분석 및 옥타브 매칭 ===")
    frame_period = 5.0

    _f0, t = pw.dio(y_processed, fs, frame_period=frame_period)
    f0_source = pw.stonemask(y_processed, _f0, t, fs)
    sp = pw.cheaptrick(y_processed, f0_source, t, fs)
    ap = pw.d4c(y_processed, f0_source, t, fs)

    # 유성음 판단
    is_voiced = (f0_source > 0)

    # --- [핵심 추가] 1. 내 목소리의 평균 음역대 구하기 ---
    if np.sum(is_voiced) > 0:
        avg_voice_f0 = np.mean(f0_source[is_voiced])
    else:
        avg_voice_f0 = 220.0 # 기본값
    print(f"-> 내 목소리 평균 Pitch: {avg_voice_f0:.2f} Hz")

    # MIDI 피치 매핑
    n_frames = len(f0_source)
    f0_midi = np.zeros(n_frames)

    # MIDI 노트 수집
    midi_notes = []
    for instrument in midi_data.instruments:
        if not instrument.is_drum:
            for note in instrument.notes:
                start_idx = int(note.start / (frame_period / 1000.0))
                end_idx = int(note.end / (frame_period / 1000.0))
                start_idx = max(0, start_idx)
                end_idx = min(n_frames, end_idx)

                note_hz = pretty_midi.note_number_to_hz(note.pitch)
                f0_midi[start_idx:end_idx] = note_hz
                midi_notes.append(note_hz)

    # --- [핵심 추가] 2. MIDI 옥타브 자동 조정 ---
    # MIDI의 평균 높이가 내 목소리와 너무 차이나면 옥타브를 이동시킴
    if len(midi_notes) > 0:
        avg_midi_f0 = np.mean(midi_notes)
        print(f"-> MIDI 원본 평균 Pitch: {avg_midi_f0:.2f} Hz")

        # 주파수 비율 계산
        ratio = avg_voice_f0 / avg_midi_f0
        # 옥타브 차이 계산 (log2)
        octave_diff = np.round(np.log2(ratio))

        if octave_diff != 0:
            scale_factor = 2 ** octave_diff
            print(f"⚠️ 경고: 음역대 차이 감지됨! {int(octave_diff)} 옥타브 조절합니다.")
            print(f"   (보정: {avg_midi_f0:.2f}Hz -> {avg_midi_f0 * scale_factor:.2f}Hz)")

            # MIDI 전체를 내 목소리 높이로 이동
            f0_midi[f0_midi > 0] *= scale_factor
        else:
            print("-> 음역대가 적절하여 옥타브 조정 없음.")

    # 최종 합성용 F0 생성
    f0_final = np.zeros_like(f0_midi)

    # 기본적으로 MIDI를 따르되, MIDI가 없는 구간은 무음 혹은 원래 목소리(선택)
    # 여기서는 노래가 목적이므로 유성음 구간에 MIDI를 강제 적용
    f0_final[is_voiced] = f0_midi[is_voiced]

    # MIDI 데이터가 끊겨서 0인 구간은 원래 목소리로 부드럽게 채움 (뚝 끊김 방지)
    missing_midi_mask = is_voiced & (f0_midi == 0)
    f0_final[missing_midi_mask] = f0_source[missing_midi_mask]

    # --- [기존 기능 유지] 스파이크/치찰음 제거 (AP=0) ---
    ap_clean = ap.copy()
    sp_clean = sp.copy()

    n_freq_bins = ap.shape[1]
    cutoff_bin = int((4000 / (fs / 2)) * n_freq_bins)

    high_freq_energy = np.mean(ap[:, cutoff_bin:], axis=1)
    is_spike_frame = (~is_voiced) & (high_freq_energy > 0.25)

    ap_clean[is_spike_frame, :] = 0.0
    sp_clean[is_spike_frame, :] *= 0.2
    ap_clean[~is_spike_frame, cutoff_bin:] *= 0.1

    print("=== [3] 합성 및 저장 ===")
    y_out = pw.synthesize(f0_final, sp_clean, ap_clean, fs, frame_period=frame_period)

    # 후처리 필터
    y_out_final = lowpass_filter(y_out, cutoff=2500, fs=fs) # 컷오프를 조금 여유있게 조정

    sf.write(output_path, y_out_final, fs)
    print(f"✅ 변환 완료: {output_path}")

# 실행 경로 설정
my_voice = "C:/rvc/preprocessing/gemini/내 꿈은 파티시엘(vocal 2).wav"
mid = "C:/rvc/preprocessing/내_꿈은_파티시엘.mid"
output = "C:/rvc/preprocessing/gemini/___________final_output222222.wav"

process_sts_safe(my_voice, mid, output)

#결과 듣기

In [ ]:
from IPython.display import Audio, display
display(Audio(output, autoplay=True))